# proventhru: M1 in Colab

M1 (`PROTOCOL.md` v2) is `Qwen/Qwen2.5-3B-Instruct`, pinned by revision hash, in fp16 under vLLM, against coqtop with **Coq 8.18.0**. The run checks the prover version, so a run on any other Coq is refused.

Runtime: **T4 GPU**. Run the cells in order. Cells 1–3 are setup. The first session takes about 20–25 minutes, because the opam build is cached to Drive; later sessions take about 3.

What this notebook may do under the protocol now: run model policies on the **dev** set (prompt tuning). Held-out runs are refused until the freezing amendment registers the prompt hash and the model revision printed by cell 4.

If installing vLLM (cell 5) replaces Colab's PyTorch and asks for a runtime restart, restart and rerun from cell 1. The cached Coq unpacks in about a minute.

In [ ]:
# 1. Drive, and where everything persistent lives
from google.colab import drive
drive.mount('/content/drive')
import os, subprocess
ROOT = '/content/drive/MyDrive/proventhru'
for d in ('', 'opam', 'runs', 'cache'):
    os.makedirs(f'{ROOT}/{d}', exist_ok=True)
BRANCH = 'experiment-protocol'   # the branch this notebook belongs to

In [ ]:
%%bash -s "$ROOT"
# 2. Coq 8.18.0 with opam. Built once, then cached on Drive as a tarball:
#    Drive is too slow to hold an opam root directly, so it is unpacked to local disk.
set -e
ROOT="$1"; TAR="$ROOT/opam/opam-coq-8.18.0.tar.zst"
apt-get -qq update >/dev/null && apt-get -qq install -y opam zstd >/dev/null
if [ -f "$TAR" ]; then
  echo "unpacking cached opam switch"; tar -C / --zstd -xf "$TAR"
else
  echo "building Coq 8.18.0 (about 15-20 minutes, once)"
  opam init -y --bare --disable-sandboxing -q
  opam switch create coq818 4.14.1 -y -q
  opam install -y -q --switch coq818 coq.8.18.0
  tar -C / --zstd -cf "$TAR" root/.opam
fi
eval "$(opam env --switch coq818 --set-switch)"
coqtop -v | head -n 1

In [ ]:
# 3. proventhru at this branch, Coq on PATH, and the coqtop tests as a smoke check
import os, subprocess
opam_bin = subprocess.run(['opam', 'var', 'bin', '--switch', 'coq818'], capture_output=True, text=True).stdout.strip()
os.environ['PATH'] = opam_bin + ':' + os.environ['PATH']
if not os.path.exists('/content/proventhru'):
    subprocess.run(['git', 'clone', '-q', '-b', BRANCH, 'https://github.com/bigDSanalyst/proventhru', '/content/proventhru'], check=True)
%cd /content/proventhru
!git log -1 --format='proventhru commit %H'
!pip -q install -e .
!coqtop -v | head -n 1
!python -m unittest tests.test_experiment 2>&1 | tail -n 3
!python -m proventhru.cli protocol check PROTOCOL.md | head -n 1

In [ ]:
# 4. M1's weights, pinned: resolve the revision once, then always load that revision.
from huggingface_hub import HfApi, snapshot_download
MODEL = 'Qwen/Qwen2.5-3B-Instruct'
REV_FILE = f'{ROOT}/m1_revision.txt'
if os.path.exists(REV_FILE):
    REVISION = open(REV_FILE).read().strip()
else:
    REVISION = HfApi().model_info(MODEL).sha
    open(REV_FILE, 'w').write(REVISION)
WEIGHTS = snapshot_download(MODEL, revision=REVISION)
SERVED = f'{MODEL}@{REVISION}'
print('M1 revision:', REVISION)
print('served as  :', SERVED, '  <- this goes into the freezing amendment')

In [ ]:
# 5. vLLM, pinned, serving M1 in fp16 (a T4 has no bf16)
VLLM = '0.6.6.post1'   # pinned; recorded in every run's identity below
!pip -q install vllm=={VLLM}
import subprocess, time, urllib.request, json
server = subprocess.Popen(
    ['python', '-m', 'vllm.entrypoints.openai.api_server', '--model', WEIGHTS,
     '--served-model-name', SERVED, '--dtype', 'float16', '--max-model-len', '4096',
     '--seed', '0', '--gpu-memory-utilization', '0.90', '--port', '8000'],
    stdout=open('/content/vllm.log', 'w'), stderr=subprocess.STDOUT)
for _ in range(180):
    try:
        print(json.load(urllib.request.urlopen('http://localhost:8000/v1/models'))['data'][0]['id']); break
    except Exception:
        time.sleep(5)
else:
    print(open('/content/vllm.log').read()[-3000:])
GPU = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], capture_output=True, text=True).stdout.strip()
PROVIDER = f'vllm {VLLM}, fp16, {GPU}'
print(PROVIDER)

In [ ]:
# 6a. Smoke test: 10 calls, one per dev statement, at the root state. No search, no record.
#     Answered? Parses against the schema? Candidates in the vocabulary? Tokens per call?
#     If json_schema fails on this vLLM, rerun with --response-format json_object and use that everywhere.
!python tools/smoke.py --base-url http://localhost:8000/v1 --model "{SERVED}" --key-env '' \
    --response-format json_schema --n 10 --protocol PROTOCOL.md --json "{ROOT}/smoke-m1.json"


In [ ]:
# 6b. One dev statement searched end to end: the schema-constrained answer comes back and is checked.
#    If json_schema is not supported by this vLLM, use --response-format json_object instead
#    (and use that same setting for every run: it is part of the policy's identity).
!python -m proventhru.cli --backend coqtop prove "forall n : nat, n * n >= n" --budget 0 --step-budget 60 \
    --policy openai --base-url http://localhost:8000/v1 --key-env '' --model "{SERVED}" \
    --response-format json_schema --provider "{PROVIDER}" --protocol PROTOCOL.md --trace | tail -n 15

In [ ]:
# 7. Dev set, 600 steps, 8 statements at once (vLLM batches their calls).
#    Prompt tuning happens here and only here. The cache makes reruns free; the run resumes if interrupted.
OUT = f'{ROOT}/runs/dev-C600'
!python -m proventhru.cli --backend coqtop run examples/eval_open.txt --out "{OUT}" --budget 0 --step-budget 600 \
    --jobs 8 --policy openai --base-url http://localhost:8000/v1 --key-env '' --model "{SERVED}" \
    --response-format json_schema --provider "{PROVIDER}" --cache "{ROOT}/cache/m1.jsonl" --protocol PROTOCOL.md
!python -m proventhru.cli report "{OUT}/records.jsonl"

## Held-out runs (after the freezing amendment)

Once the prompt is frozen and `PROTOCOL.md` registers `SERVED` and the prompt hash, the runs are C and D on `test`, and C on `test_renamed`, each at `--step-budget 600`. The baselines A and B are rerun on `test` under the same version. Until then these commands are refused by the protocol check, which is intended.

In [ ]:
# for c, extra in (('C600', ''), ('D600', '--retrieval 6')):
#     !python -m proventhru.cli --backend coqtop run examples/eval_test.txt --out "{ROOT}/runs/test-{c}" --budget 0 \
#         --step-budget 600 --jobs 8 --policy openai --base-url http://localhost:8000/v1 --key-env '' --model "{SERVED}" \
#         --response-format json_schema --provider "{PROVIDER}" --cache "{ROOT}/cache/m1.jsonl" --protocol PROTOCOL.md {extra}